# Exercices

## **Préliminaires**: Clone de votre repo et imports

In [6]:
! git clone https://github.com/medz1966/exam_2025.git
! cp exam_2025/utils/utils_exercices.py .

import copy
import numpy as np
import torch
import torch.nn as nn

fatal: destination path 'exam_2025' already exists and is not an empty directory.


**Clef personnelle pour la partie théorique**

Dans la cellule suivante, choisir un entier entre 100 et 1000 (il doit être personnel). Cet entier servira de graine au générateur de nombres aléatoire a conserver pour tous les exercices.



In [2]:
mySeed = 200

\

---

\

\

**Exercice 1** *Une relation linéaire*

La fonction *generate_dataset* fournit deux jeux de données (entraînement et test). Pour chaque jeu de données, la clef 'inputs' donne accès à un tableau numpy (numpy array) de prédicteurs empilés horizontalement : chaque ligne $i$ contient trois prédicteurs $x_i$, $y_i$ et $z_i$. La clef 'targets' renvoie le vecteur des cibles $t_i$. \

Les cibles sont liées aux prédicteurs par le modèle:
$$ t = \theta_0 + \theta_1 x + \theta_2 y + \theta_3 z + \epsilon$$ où $\epsilon \sim \mathcal{N}(0,\eta)$


In [3]:
from utils_exercices import generate_dataset, Dataset1
train_set, test_set = generate_dataset(mySeed)

[texte du lien](https://)**Q1** Par quelle méthode simple peut-on estimer les coefficients $\theta_k$ ? La mettre en oeuvre avec la librairie python de votre choix.


Pour estimer les coefficients $\theta$, nous utilisons la méthode des moindres carrés. Cette méthode minimise la somme des erreurs quadratiques :

$$
\min ||X\theta - y||^2
$$

où $X$ est la matrice des prédicteurs et $y$ est le vecteur des cibles. La solution analytique est donnée par :

$$
\theta = (X^T X)^{-1} X^T y
$$

In [4]:
from numpy.linalg import inv

# Données d'entraînement
X = train_set['inputs']
y = train_set['targets']

# Ajout d'une colonne de biais (1)
X_b = np.hstack([X, np.ones((X.shape[0], 1))])

# Calcul de \theta
theta_hat = inv(X_b.T @ X_b) @ X_b.T @ y
print("Coefficients estimés (theta):", theta_hat)


Coefficients estimés (theta): [ 1.95156862  1.94842221  3.59966699 10.07876403]


**Q2** Dans les cellules suivantes, on se propose d'estimer les $\theta_k$ grâce à un réseau de neurones entraîné par SGD. Quelle architecture s'y prête ? Justifier en termes d'expressivité et de performances en généralisation puis la coder dans la cellule suivante.

Pour estimer les coefficients $\theta_k$, une architecture simple comme un réseau de neurones à une seule couche entièrement connectée est appropriée, car un réseau entièrement connecté avec une couche linéaire est suffisant pour modéliser une relation linéaire. Ainsi, une architecture simple limite le risque de sur-apprentissage tout en assurant une bonne capacité d'apprentissage sur les données.


In [7]:
# Dataset et dataloader :
dataset = Dataset1(train_set['inputs'], train_set['targets'])
dataloader = torch.utils.data.DataLoader(dataset, batch_size=100, shuffle=True)

# A coder :
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Une seule couche linéaire : 3 entrées, 1 sortie
        self.fc = nn.Linear(3, 1)

    def forward(self, x):
        # Application de la couche linéaire
        return self.fc(x)


**Q3** Entraîner cette architecture à la tâche de régression définie par les entrées et sorties du jeu d'entraînement (compléter la cellule ci-dessous).

In [8]:
# Initialize model, loss, and optimizer
mySimpleNet = SimpleNet()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(mySimpleNet.parameters(), lr=0.01)

# Training loop
num_epochs = 500
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in dataloader:
        # Forward pass : calcul des prédictions
        predictions = mySimpleNet(batch_inputs)

        # Calcul de la perte
        loss = criterion(predictions, batch_targets)

        # Backward pass : calcul du gradient
        optimizer.zero_grad()
        loss.backward()

        # Mise à jour des poids
        optimizer.step()

    # Afficher la perte toutes les 50 époques
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([100])) that is different to the input size (torch.Size([100, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 50/500, Loss: 12.1462
Epoch 100/500, Loss: 13.9852
Epoch 150/500, Loss: 13.1347
Epoch 200/500, Loss: 16.1897
Epoch 250/500, Loss: 15.9822
Epoch 300/500, Loss: 13.3497
Epoch 350/500, Loss: 13.0254
Epoch 400/500, Loss: 15.7697
Epoch 450/500, Loss: 16.2812
Epoch 500/500, Loss: 16.4888


**Q4** Où sont alors stockées les estimations des  $\theta_k$ ? Les extraire du réseau *mySimpleNet* dans la cellule suivante.

Les estimations des coefficients $\theta_k$ sont stockées dans les paramètres de la couche linéaire du réseau, spécifiquement dans :
- Les poids (\texttt{weight}) de la couche linéaire.
- Le biais (\texttt{bias}) de la couche linéaire.

In [10]:
weights = mySimpleNet.fc.weight.data.numpy()  # Les poids (theta_1, theta_2, theta_3)
bias = mySimpleNet.fc.bias.data.numpy()       # Le biais (theta_0)

print("Coefficients estimés (poids) :", weights)
print("Biais estimé :", bias)

Coefficients estimés (poids) : [[0.01971598 0.01649693 0.03512505]]
Biais estimé : [11.912238]


**Q5** Tester ces estimations sur le jeu de test et comparer avec celles de la question 1. Commentez.

In [13]:
# prompt: Q5 Tester ces estimations sur le jeu de test et comparer avec celles de la question 1. Commentez

# Prédictions sur le jeu de test avec les coefficients estimés par le modèle linéaire (Q1)
X_test = test_set['inputs']
y_test = test_set['targets']
X_test_b = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
y_pred_q1 = X_test_b @ theta_hat

# Prédictions sur le jeu de test avec le réseau de neurones (Q4)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_pred_q4 = mySimpleNet(X_test_tensor).detach().numpy().flatten()

# Comparaison des performances (MSE)
mse_q1 = np.mean((y_test - y_pred_q1)**2)
mse_q4 = np.mean((y_test - y_pred_q4)**2)

print(f"MSE (Question 1) : {mse_q1}")
print(f"MSE (Question 4) : {mse_q4}")

print("Comparaison :")
if mse_q1 < mse_q4:
    print("La méthode des moindres carrés (Q1) a de meilleures performances sur le jeu de test.")
elif mse_q1 > mse_q4:
    print("Le réseau de neurones (Q4) a de meilleures performances sur le jeu de test.")
else:
    print("Les deux méthodes ont des performances similaires sur le jeu de test.")

MSE (Question 1) : 4.0092609424969
MSE (Question 4) : 14.49096874140609
Comparaison :
La méthode des moindres carrés (Q1) a de meilleures performances sur le jeu de test.


Les réseaux de neurones sont adaptés aux relations complexes, mais peuvent sur-ajuster les données simples comme ici, entraînant une MSE plus élevée.

La régression linéaire, conçue pour des relations linéaires, offre une solution optimale avec une MSE plus faible.


\

---

\

**Exercice 2** *Champ réceptif et prédiction causale*

Le réseau défini dans la cellule suivante est utilisé pour faire le lien entre les valeurs $(x_{t' \leq t})$ d'une série temporelle d'entrée et la valeur présente $y_t$ d'une série temporelle cible.

In [14]:
import torch.nn as nn
import torch.nn.functional as F
from utils_exercices import Outconv, Up_causal, Down_causal

class Double_conv_causal(nn.Module):
    '''(conv => BN => ReLU) * 2, with causal convolutions that preserve input size'''
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1):
        super(Double_conv_causal, self).__init__()
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=0, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size, padding=0, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)

    def forward(self, x):
        x = F.pad(x, ((self.kernel_size - 1) * self.dilation, 0))
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = F.pad(x, ((self.kernel_size - 1) * self.dilation, 0))
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x


class causalFCN(nn.Module):
    def __init__(self, dilation=1):
        super(causalFCN, self).__init__()
        size = 64
        n_channels = 1
        n_classes = 1
        self.inc = Double_conv_causal(n_channels, size)
        self.down1 = Down_causal(size, 2*size)
        self.down2 = Down_causal(2*size, 4*size)
        self.down3 = Down_causal(4*size, 8*size, pooling_kernel_size=5, pooling_stride=5)
        self.down4 = Down_causal(8*size, 4*size, pooling=False, dilation=2)
        self.up2 = Up_causal(4*size, 2*size, kernel_size=5, stride=5)
        self.up3 = Up_causal(2*size, size)
        self.up4 = Up_causal(size, size)
        self.outc = Outconv(size, n_classes)
        self.n_classes = n_classes

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up2(x5, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        x = self.outc(x)
        return x

# Exemple d'utilisation
model = causalFCN()
# Série temporelle d'entrée (x_t):
input_tensor1 = torch.rand(1, 1, 10000)
# Série temporelle en sortie f(x_t):
output = model(input_tensor1)
print(output.shape)

torch.Size([1, 1, 10000])


**Q1** De quel type de réseau de neurones s'agit-il ? Combien de paramètres la couche self.Down1 compte-t-elle (à faire à la main) ?
Combien de paramètres le réseau entier compte-t-il (avec un peu de code) ?

**Q1 : De quel type de réseau s'agit-il ?**

Le réseau est un **Fully Convolutional Network (FCN)** à convolutions causales. Les convolutions causales garantissent que chaque sortie $y_t$ dépend uniquement des entrées $x_{t'}$ où $t' \leq t$, respectant ainsi la causalité dans les séries temporelles. Cela en fait un modèle adapté pour prédire des valeurs cibles dans des contextes séquentiels.

---

**Nombre de paramètres de la couche `self.down1` :**

- **Formule pour une couche `Conv1d`** :  
  $$ \text{Paramètres} = (\text{in_ch} \times \text{out_ch} \times \text{kernel_size}) + \text{out_ch} $$

- **Formule pour `BatchNorm1d`** :  
  $$ \text{Paramètres} = 2 \times \text{out_ch} $$

Pour `self.down1` :  
- Entrées : $in\_ch = 64$, sorties : $out\_ch = 128$, taille du kernel : $kernel\_size = 3$  
  $$ \text{Conv1d} : (64 \times 128 \times 3) + 128 = 24672 $$  
  $$ \text{BatchNorm1d} : 2 \times 128 = 256 $$  
  **Total** :  
  $$ 24672 + 256 = 24928 \text{ paramètres.} $$

---

**Nombre total de paramètres dans le réseau :**

D'après le calcul, le réseau contient :  
$$ \text{Nombre total de paramètres} = 2872641 $$

---

In [18]:
# Inspect internal structure of self.down1
print(model.down1)

# Calculate parameters for each layer in self.down1
for name, param in model.down1.named_parameters():
    print(f"Layer: {name}, Parameters: {param.numel()}")


Down_causal(
  (mpconv): Sequential(
    (0): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (1): Double_conv_causal(
      (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,))
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
)
Layer: mpconv.1.conv1.weight, Parameters: 24576
Layer: mpconv.1.conv1.bias, Parameters: 128
Layer: mpconv.1.bn1.weight, Parameters: 128
Layer: mpconv.1.bn1.bias, Parameters: 128
Layer: mpconv.1.conv2.weight, Parameters: 49152
Layer: mpconv.1.conv2.bias, Parameters: 128
Layer: mpconv.1.bn2.weight, Parameters: 128
Layer: mpconv.1.bn2.bias, Parameters: 128


In [17]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Nombre total de paramètres : {total_params}")


Nombre total de paramètres : 2872641


**Q2** Par quels mécanismes la taille du vecteur d'entrée est-elle réduite ? Comment est-elle restituée dans la deuxième partie du réseau ?

### Réduction de la taille du vecteur d'entrée

La taille du vecteur d'entrée est réduite dans la première partie du réseau par les mécanismes suivants :

**Pooling dans les couches `Down_causal` :**
   - Les couches de pooling (comme `MaxPool1d`) réduisent la longueur du vecteur d'entrée en prenant des sous-échantillons de la séquence.
   - Par exemple, un `MaxPool1d` avec un `kernel_size=2` et un `stride=2` divise la longueur par 2.


---

### Restitution de la taille dans la deuxième partie du réseau

La taille du vecteur est restaurée dans la deuxième partie du réseau grâce aux mécanismes suivants :

1. **Upsampling dans les couches `Up_causal` :**
   - Ces couches utilisent des convolutions transposées ou des opérations similaires pour augmenter la longueur du vecteur.
   - Par exemple, un `Up_causal` avec `kernel_size=5` et `stride=5` multiplie la longueur par 5.

2. **Concaténation avec les sorties intermédiaires :**
   - Pendant la phase de remontée (upsampling), les sorties intermédiaires des couches descendantes (`down1`, `down2`, etc.) sont concaténées avec les sorties des couches montantes correspondantes (`up2`, `up3`, etc.).
   - Cela permet de récupérer les informations perdues lors de la réduction de taille et améliore la précision.

---

**Q3** Par quels mécanismes le champ réceptif est-il augmenté ? Préciser par un calcul la taille du champ réceptif en sortie de *self.inc*.

### Mécanismes d'augmentation du champ réceptif

Le champ réceptif est augmenté par les mécanismes suivants :

1. **Convolutions avec des kernels de grande taille :**
   - Chaque couche de convolution étend le champ réceptif en intégrant les informations des entrées sur plusieurs pas de temps.

2. **Dilatation (dilated convolutions) :**
   - La dilatation insère des "trous" entre les valeurs prises en compte par le kernel, augmentant ainsi l'intervalle couvert sans augmenter le nombre de paramètres.

3. **Empilement des couches convolutionnelles :**
   - Chaque couche successive augmente le champ réceptif, car elle dépend des sorties de la couche précédente, qui elles-mêmes dépendent d'une plage élargie de l'entrée.
### Calcul du champ réceptif pour `self.inc`

La couche `self.inc` est composée de deux convolutions successives (`conv1` et `conv2`), chacune avec :
- **Taille du kernel** : kernel_size = 3 ,
- **Dilatation** : dilation = 1 \.

#### Formule générale pour une convolution :
\[
champ_réceptif = ( kernel_size - 1 )* (dilation + 1)
\]

#### Étape 1 : Champ réceptif de `conv1` :

champ\_réceptif_conv1 = (3 - 1) * 1 + 1 = 3

#### Étape 2 : Champ réceptif de `conv2` :
- Le champ réceptif de `conv2` dépend de celui de `conv1` :
\[
champ\_réceptif_conv2 = (champ\_réceptif_conv1 - 1) + ((3 - 1) * 1 + 1) = 5
\]s

**Résultat :**
Le champ réceptif total de `self.inc` est de **5**.


In [20]:
# prompt: Q3 Par quels mécanismes le champ réceptif est-il augmenté ? Préciser par un calcul la taille du champ réceptif en sortie de self.inc.

# Q3: Mécanismes d'augmentation du champ réceptif

# Le champ réceptif est augmenté principalement par :

# 1. Dilatation des convolutions :
#    - Augmente la taille du champ réceptif sans augmenter le nombre de paramètres,
#    - Permet à la convolution de "voir" plus loin dans le signal d'entrée sans chevauchement des kernels.

# 2. Empilement des couches convolutives :
#    - Chaque couche convolutive élargit le champ réceptif.
#    - Avec un empilement, le champ réceptif augmente exponentiellement.


# Calcul du champ réceptif en sortie de self.inc :

# Dans self.inc, il y a deux convolutions 1D avec la même dilatation.
kernel_size = 3
dilation = 1

# Champ receptif d'une seule convolution 1D
rf_one_conv = kernel_size

# Champ receptif pour deux convolutions 1D consécutives
rf_double_conv = 1 + 2 * (kernel_size-1)

print("Champ réceptif en sortie de self.inc:", rf_double_conv)

Champ réceptif en sortie de self.inc: 5


**Q4** Par un bout de code, déterminer empiriquement la taille du champ réceptif associé à la composante $y_{5000}$ du vecteur de sortie. (Indice: considérer les sorties associées à deux inputs qui ne diffèrent que par une composante...)

In [24]:
import torch
import copy

# Instantiate the model
model = causalFCN()

# Create two identical input tensors
input_tensor1 = torch.zeros(1, 1, 10000)  # Input sequence of length 10,000
input_tensor2 = input_tensor1.clone()

# Modify one value in the second tensor
modification_index = 5000  # Target index for output y5000
input_tensor2[0, 0, modification_index] = 1.0

# Get model outputs for both inputs
output1 = model(input_tensor1)
output2 = model(input_tensor2)

# Compute the absolute difference between outputs
diff = torch.abs(output1 - output2)

# Identify the affected indices in the output
affected_indices = torch.nonzero(diff[0, 0]).squeeze()

# Calculate the receptive field size
receptive_field_start = affected_indices.min().item()
receptive_field_end = affected_indices.max().item()
receptive_field_size = receptive_field_end - receptive_field_start + 1

print("Champ réceptif empirique pour y5000 :", receptive_field_size)


Champ réceptif empirique pour y5000 : 10000


**Q5** $y_{5000}$ dépend-elle des composantes $x_{t, \space t > 5000}$ ? Justifier de manière empirique puis préciser la partie du code de Double_conv_causal qui garantit cette propriété de "causalité" en justifiant.  



Pour vérifier si $y_{5000}$ dépend des composantes $x_t$ pour $t > 5000$, nous testons empiriquement en modifiant uniquement les valeurs d’entrée pour $t > 5000$ et en observant l’impact sur la sortie $y_{5000}$.



In [28]:
# Créer deux tenseurs d'entrée identiques jusqu'à l'index 5000
input_tensor1 = torch.rand(1, 1, 10000)
input_tensor2 = input_tensor1.clone()

# Modifier une composante après l'index 5000 dans le second tenseur
modification_index = 5001  # Index de la modification (ex: 5001)
input_tensor2[0, 0, modification_index] = 1.0  # Modifier la valeur

# Obtenir les sorties du modèle
output1 = model(input_tensor1)
output2 = model(input_tensor2)

# Comparer les sorties à l'index 5000
diff_at_5000 = torch.abs(output1[0, 0, 5000] - output2[0, 0, 5000])

print(f"Différence à l'index 5000 : {diff_at_5000}")

if diff_at_5000.item() == 0 :
  print("y_5000 ne dépend pas des composantes x_t pour t > 5000")
else:
  print("y_5000 dépend des composantes x_t pour t > 5000")


Différence à l'index 5000 : 0.011476337909698486
y_5000 dépend des composantes x_t pour t > 5000


\

---

\

\

Exercice 3: "Ranknet loss"

Un [article récent](https://https://arxiv.org/abs/2403.14144) revient sur les progrès en matière de learning to rank. En voilà un extrait :


<img src="https://raw.githubusercontent.com/nanopiero/exam_2025/refs/heads/main/utils/png_exercice3.PNG?token=GHSAT0AAAAAAC427DACOPGNDNN6UDOLVLLAZ4BB2JQ" alt="extrait d'un article" width="800">

**Q1** Qu'est-ce que les auteurs appellent "positive samples" et "negative samples" ? Donner un exemple.

Les auteurs appellent **"positive samples"** les exemples où un utilisateur a cliqué sur une annonce (click = 1) et **"negative samples"** ceux où l'utilisateur n'a pas cliqué (click = 0).  

**Exemple :**  
- **Positive sample** : Une annonce a été cliquée par un utilisateur.  
- **Negative sample** : Une annonce a été ignorée par un utilisateur.

**Q2** Dans l'expression de $\mathcal{L}_{RankNet}$, d'où proviennent les $z_i$ ? Que représentent-ils ?  

### **Formule de \( L_{RankNet} \)** :
$$
L_{RankNet} = - \frac{1}{N^+ N^-} \sum_{i=1}^{N^+} \sum_{j=1}^{N^-} \log(\sigma(z_i^{(+)} - z_j^{(-)}))
$$

---

### **Définition des termes** :
1. **\( z_i^{(+)} \) et \( z_j^{(-)} \)** :
   - $$ z_i^{(+)} $$ : Logit associé à un **exemple positif** (par ex. une annonce cliquée).
   - $$ z_j^{(-)} $$ : Logit associé à un **exemple négatif** (par ex. une annonce non cliquée).

2. **\( \sigma \)** :
   - $$ \sigma(x) = \frac{1}{1 + e^{-x}} $$ : Fonction sigmoïde qui transforme les différences $$ z_i^{(+)} - z_j^{(-)} $$ en probabilités, indiquant si $$ z_i^{(+)} $$ est "plus grand" que $$ z_j^{(-)} $$.

3. **\( N^+ \) et \( N^- )** :
   - $$ N^+ $$ : Nombre d'exemples positifs.
   - $$ N^- $$ : Nombre d'exemples négatifs.

---



- Les $$ z_i $$ sont les **valeurs calculées par le modèle** (logits) pour chaque exemple.
- $$ z_i^{(+)} $$ et $$ z_j^{(-)} $$ permettent de comparer un exemple positif et un exemple négatif.
- $$ L_{RankNet} $$ pousse le modèle à donner des **valeurs plus élevées (logits)** aux exemples positifs qu'aux négatifs, afin d'améliorer le classement.


**Q3** Pourquoi cette expression conduit-elle à ce que, après apprentissage, "the estimated
value of positive samples is greater than that of negative samples
for each pair of positive/negative samples" ?


### **Formule de \( L_{RankNet} \)** :
$$
L_{RankNet} = - \frac{1}{N^+ N^-} \sum_{i=1}^{N^+} \sum_{j=1}^{N^-} \log(\sigma(z_i^{(+)} - z_j^{(-)}))
$$

---

1. **La fonction sigmoïde** :
   - La sigmoïde \( \sigma(x) \) est définie par :
     $$
     \sigma(x) = \frac{1}{1 + e^{-x}}
     $$
   - Lorsque \( z_i^{(+)} > z_j^{(-)} \), \($$ \sigma(z_i^{(+)} - z_j^{(-)}) $$) s'approche de 1. Cela minimise la perte \( L_{RankNet} \).

2. **Minimisation de la perte** :
   - Pendant l'apprentissage, le modèle ajuste les valeurs \( z_i^{(+)} \) (exemples positifs) et \( z_j^{(-)} \) (exemples négatifs) pour réduire \( L_{RankNet} \).
   - La réduction de \( L_{RankNet} \) est possible lorsque \( z_i^{(+)} > z_j^{(-)} \) pour tous les couples positif/négatif.


**Q4** Dans le cadre d'une approche par deep learning, quels termes utilise-t-on pour qualifier les réseaux de neurones exploités et la modalité suivant laquelle ils sont entraînés ?



- **Réseaux utilisés** : Réseaux entièrement connectés (FCNNs) adaptés aux données tabulaires.
- **Modalités d'entraînement** :
  - **Pointwise** : Exemple traité individuellement (perte BCE).
  - **Pairwise** : Comparaison d'exemples positifs/négatifs (perte \( L_{RankNet} \)).
  - **Listwise** : Optimisation de l'ordre complet des classements (perte ListNet ou similaires).